# RNA Velocity

Launcher na kernelu **`velo-core`** (Fig. 6D z Luo et al. 2026):

1. **Inference** — metody odpalane przez CLI w `src/rna_velocity/scripts/` (właściwy conda env per metoda; wynik: `.h5ad` z `layers['velocity']`).
2. **Downstream** — wspólny CellRank na zapisanych wynikach (lokalnie w tym notebooku).

Obce metody (DeepVelo, UniTVelo, LatentVelo, Pyro-Velocity) **nie** są importowane tutaj — tylko `subprocess` / `conda run`.

## Imports

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE" #macOS only fix

In [3]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import anndata as ad
import cellrank as cr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv

# Repo root → src on path (CLI runners use PYTHONPATH=src the same way)
ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from rna_velocity.registry import METHOD_ENV, METHOD_VKEY

sc.settings.verbosity = 2
scv.settings.verbosity = 2
print("methods:", list(METHOD_ENV))
print("vkeys:", METHOD_VKEY)

methods: ['scvelo_stochastic', 'dynamo', 'velovi', 'deepvelo', 'unitvelo', 'latentvelo', 'pyrovelocity']
vkeys: {'scvelo_stochastic': 'velocity', 'dynamo': 'velocity_S', 'velovi': 'velocity', 'deepvelo': 'velocity', 'unitvelo': 'velocity', 'latentvelo': 'spliced_velocity', 'pyrovelocity': 'velocity_pyro'}


## Helper functions

In [4]:
def check_velocity_adata(
    adata: ad.AnnData,
    *,
    stage: str = "inference",
    cluster_key: str = "clusters",
    require_clusters: bool = False,
    require_umap: bool = False,
    min_nonzero_frac: float = 0.0,
    raise_on_fail: bool = False,
    verbose: bool = True,
) -> dict:
    """Validate AnnData against the RNA-velocity contract.

    stage="inference"  — needs layers['spliced'] / layers['unspliced'] (CLI input)
    stage="downstream" — needs layers['velocity'] (+ neighbors for CellRank)

    Returns a report dict with ok / missing / warnings. Optionally raises.
    """
    if stage not in {"inference", "downstream"}:
        raise ValueError(f"stage must be 'inference' or 'downstream', got {stage!r}")

    missing: list[str] = []
    warnings: list[str] = []
    details: dict = {"n_obs": adata.n_obs, "n_vars": adata.n_vars, "stage": stage}

    def _layer_ok(name: str) -> bool:
        if name not in adata.layers:
            missing.append(f"layers[{name!r}]")
            return False
        layer = adata.layers[name]
        shape = getattr(layer, "shape", None)
        if shape != adata.shape:
            missing.append(f"layers[{name!r}] shape {shape} != adata.shape {adata.shape}")
            return False
        if min_nonzero_frac > 0:
            if hasattr(layer, "nnz"):
                frac = layer.nnz / (adata.n_obs * adata.n_vars)
            else:
                arr = np.asarray(layer)
                frac = float((arr != 0).mean())
            details[f"{name}_nonzero_frac"] = frac
            if frac < min_nonzero_frac:
                warnings.append(
                    f"layers[{name!r}] nonzero frac {frac:.4g} < min_nonzero_frac={min_nonzero_frac}"
                )
        return True

    if stage == "inference":
        _layer_ok("spliced")
        _layer_ok("unspliced")
    else:
        _layer_ok("velocity")
        has_neighbors = (
            "neighbors" in adata.uns
            or "connectivities" in adata.obsp
            or "distances" in adata.obsp
        )
        details["has_neighbors"] = has_neighbors
        if not has_neighbors:
            missing.append("neighbors (uns['neighbors'] or obsp connectivities/distances)")

    if require_clusters or cluster_key in adata.obs:
        if cluster_key not in adata.obs:
            missing.append(f"obs[{cluster_key!r}]")
        else:
            details["n_clusters"] = int(adata.obs[cluster_key].nunique())
    elif stage == "downstream":
        warnings.append(
            f"obs[{cluster_key!r}] absent — set require_clusters=True if CellRank needs labels"
        )

    if require_umap and "X_umap" not in adata.obsm:
        missing.append("obsm['X_umap']")
    elif "X_umap" not in adata.obsm:
        warnings.append("obsm['X_umap'] absent (needed for stream / embedding plots)")

    # hint: splicing under alternate names
    if stage == "inference" and missing:
        alt = [
            k
            for k in adata.layers
            if "splic" in k.lower() or "unsplic" in k.lower()
        ]
        if alt:
            warnings.append(f"related layer keys found (not standard names): {alt}")

    ok = len(missing) == 0
    report = {
        "ok": ok,
        "missing": missing,
        "warnings": warnings,
        "details": details,
        "layers": list(adata.layers.keys()),
    }

    if verbose:
        status = "OK" if ok else "FAIL"
        print(f"[{status}] stage={stage}  shape={adata.shape}")
        if missing:
            print("  missing:")
            for m in missing:
                print(f"    - {m}")
        if warnings:
            print("  warnings:")
            for w in warnings:
                print(f"    - {w}")
        if ok and not warnings:
            print("  all required fields present")

    if raise_on_fail and not ok:
        raise ValueError(
            f"AnnData failed velocity {stage} check: missing {missing}"
        )
    return report


def run_velocity_cli(
    method: str,
    input_path: Path,
    output_path: Path,
    *,
    cluster_key: str = "clusters",
    extra_args: list[str] | None = None,
) -> Path:
    """Launch `scripts/run_<method>.py` in METHOD_ENV[method] via conda run.

    Blocks until the process exits. Does not run CellRank / downstream.
    """
    if method not in METHOD_ENV:
        raise KeyError(f"unknown method {method!r}; choose from {list(METHOD_ENV)}")

    script = SRC / "rna_velocity" / "scripts" / f"run_{method}.py"
    if not script.is_file():
        raise FileNotFoundError(script)

    env_name = METHOD_ENV[method]
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    cmd = [
        "conda",
        "run",
        "-n",
        env_name,
        "--no-capture-output",
        "env",
        f"PYTHONPATH={SRC}",
        "KMP_DUPLICATE_LIB_OK=TRUE",
        "python",
        str(script),
        "--input",
        str(input_path),
        "--output",
        str(output_path),
        "--cluster-key",
        cluster_key,
        *(extra_args or []),
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True, cwd=ROOT)
    print(f"done → {output_path}  (env={env_name}, vkey={METHOD_VKEY[method]})")
    return output_path


## Download data

Smoke-test dataset z *sc-best-practices* / scVelo: **pancreatic endocrinogenesis** (Bastidas-Ponce et al. 2019). Ma `layers['spliced']` / `layers['unspliced']`, `obs['clusters']`, `obsm['X_umap']` — w przeciwieństwie do `merged_dataset.h5ad` z TI.

Źródło: `scv.datasets.pancreas()` (bez konta Lamin). Alternatywa Lamin: `trajectory/rna_velocity.h5ad` z `theislab/sc-best-practices` — patrz `docs/chat-notes-veloBench.md` §13.

In [5]:
DATA_DIR = ROOT / "data" / "Pancreas"
DATA_PATH = DATA_DIR / "Pancreas.h5ad"

if DATA_PATH.exists():
    print(f"already present: {DATA_PATH}")
else:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    adata_dl = scv.datasets.pancreas()
    adata_dl.write_h5ad(DATA_PATH)
    print(f"wrote {DATA_PATH}  shape={adata_dl.shape}")
    print("layers:", list(adata_dl.layers.keys()))

try downloading from url
https://github.com/theislab/scvelo_notebooks/raw/master/data/Pancreas/endocrinogenesis_day15.h5ad
... this may take a while but only happens once


  0%|          | 0.00/50.0M [00:00<?, ?B/s]

wrote /Users/jasiek/ideas/boolai/data/Pancreas/Pancreas.h5ad  shape=(3696, 27998)
layers: ['spliced', 'unspliced']


## Data load

In [6]:
adata = sc.read_h5ad(DATA_PATH)
check_velocity_adata(adata, require_clusters=True, require_umap=True)
adata

[OK] stage=inference  shape=(3696, 27998)
  all required fields present


AnnData object with n_obs × n_vars = 3696 × 27998
    obs: 'clusters_coarse', 'clusters', 'S_score', 'G2M_score'
    var: 'highly_variable_genes'
    uns: 'clusters_coarse_colors', 'clusters_colors', 'day_colors', 'neighbors', 'pca'
    obsm: 'X_pca', 'X_umap'
    layers: 'spliced', 'unspliced'
    obsp: 'connectivities', 'distances'

## Run velocity (CLI)

Inference tylko — `conda run -n {METHOD_ENV[method]}` + skrypt z `src/rna_velocity/scripts/`.  
Wynik: `results/{method}/{dataset}_velocity.h5ad` z `layers['velocity']`. Bez CellRank / fate / plotów downstream.

Domyślnie smoke: **`scvelo_stochastic`** (env `velo-core`). Zmień `METHOD`, żeby odpalić inną z Fig. 6D.

In [ ]:
METHOD = "scvelo_stochastic"  # one of METHOD_ENV keys
CLUSTER_KEY = "clusters"
DATASET = DATA_PATH.stem  # e.g. Pancreas

OUT_PATH = ROOT / "results" / METHOD / f"{DATASET}_velocity.h5ad"
METHOD, METHOD_ENV[METHOD], OUT_PATH

In [ ]:
run_velocity_cli(
    METHOD,
    DATA_PATH,
    OUT_PATH,
    cluster_key=CLUSTER_KEY,
)

In [ ]:
adata_vel = sc.read_h5ad(OUT_PATH)
check_velocity_adata(adata_vel, stage="downstream", cluster_key=CLUSTER_KEY)
print("velocity_method:", adata_vel.uns.get("velocity_method"))
print("velocity_vkey:", adata_vel.uns.get("velocity_vkey"))
adata_vel

## CellRank downstream — trajektoria z velocity

Velocity daje tylko **lokalny kierunek** (wektor), nie globalną oś czasu — CellRank sam z siebie nie ma odpowiednika `dpt_pseudotime`. Zamiast dyfuzji na transkryptomie (DPT) albo `scv.tl.velocity_pseudotime` (tylko sanity-check, nie właściwy wynik — patrz `docs/chat-notes-veloBench.md` §9), używamy tu **opcji 4** z §15: `GPCCA.compute_absorption_times()`.

Kroki:

1. `VelocityKernel` (kierunek z velocity) + `ConnectivityKernel` (stabilizacja, standardowa recepta CellRank: `0.8*vk + 0.2*ck`) → transition matrix.
2. `GPCCA`: macrostates → terminal / initial states → fate probabilities (prawdopodobieństwo trafienia do każdego terminal state).
3. `compute_absorption_times()` — oczekiwany **pozostały** czas (w krokach łańcucha Markowa) do absorpcji w każdym terminal state. To rośnie *od* terminali *do* korzenia (im dalej od mety, tym dłuższy oczekiwany czas) — odwrotnie niż `dpt_pseudotime`.
4. Agregacja do jednego skalara: `remaining = Σ_state fate_prob[state] · absorption_time[state]` (oczekiwany czas do *dowolnego* terminala, ważony tym, dokąd komórka faktycznie zmierza), potem odwrócenie kierunku i normalizacja do `[0, 1]`, żeby `pseudotime` rosło root→terminal jak w `4. Trajectory Inference.ipynb`.

Wynik zapisujemy w tych samych nazwach co kontrakt TI: `obs['pseudotime']`, `obs['branch']`, `uns['root_state']`, `uns['terminal_states']`, `uns['transient_states']`. Eksport CSV/JSON w stylu `pseudotime_inferred_observations.csv` / `transition_constraints.json` (BoNesis) — świadomie **nie** robimy tu jeszcze; to osobny krok, patrz §15.

In [ ]:
import cellrank as cr

from trajectory_inference import (
    select_observations,
    derive_bonesis_constraints,
    write_bonesis_constraints,
    write_trajectory_h5ad,
    plot_pseudotime_boxplot,
    plot_pseudotime_kde,
    plot_state_graph,
)

adata_cr = adata_vel.copy()

vk = cr.kernels.VelocityKernel(adata_cr).compute_transition_matrix()
ck = cr.kernels.ConnectivityKernel(adata_cr).compute_transition_matrix()
combined_kernel = 0.8 * vk + 0.2 * ck
combined_kernel

### Macrostates → terminal / initial states (GPCCA)

`N_MACROSTATES` steruje rozdzielczością — za mało i GPCCA złapie tylko część prawdziwych terminal states (np. na Pancreas z `n_states=6` łapie 3 z 4 znanych fenotypów terminalnych: Alpha/Beta/Delta/Epsilon); za dużo i macrostates zaczynają się dzielić na warianty tego samego typu (`Ductal_1`/`Ductal_2`). Wartość dobrana empirycznie pod smoke dataset — do przestrojenia per dataset.

In [ ]:
N_MACROSTATES = 6

g = cr.estimators.GPCCA(combined_kernel)
g.compute_macrostates(n_states=N_MACROSTATES, cluster_key=CLUSTER_KEY)
g.predict_terminal_states()
g.predict_initial_states(n_states=1)

print("macrostates:", g.macrostates.cat.categories.tolist())
print("terminal_states:", g.terminal_states.cat.categories.tolist())
print("initial_states:", g.initial_states.cat.categories.tolist())

g.plot_macrostates(which="all", basis="umap", legend_loc="right")
plt.show()

### Fate probabilities

Dla każdej komórki: prawdopodobieństwo trafienia do każdego terminal state (miękka alternatywa dla twardego przypisania do gałęzi).

In [ ]:
g.compute_fate_probabilities()
g.plot_fate_probabilities(basis="umap", same_plot=False)
plt.show()

fate_probabilities = g.fate_probabilities
terminal_names = list(fate_probabilities.names)
fate_df = pd.DataFrame(np.asarray(fate_probabilities), columns=terminal_names, index=adata_cr.obs_names)
fate_df.head()

### Pseudoczas z absorption times (opcja 4, §15)

`compute_absorption_times()` daje **pozostały** czas do absorpcji per terminal state (`{state}_mean`), rosnący *od* terminali *do* korzenia. Ważymy go fate probabilities (oczekiwany czas do *dowolnego* terminala, nie do jednego z góry wybranego), po czym odwracamy kierunek i normalizujemy do `[0, 1]`, żeby otrzymać `pseudotime` rosnący korzeń→terminal — tak jak `dpt_pseudotime` w `4. Trajectory Inference.ipynb`.

In [ ]:
g.compute_absorption_times()

absorption_df = g.absorption_times[[f"{name}_mean" for name in terminal_names]].copy()
absorption_df.columns = terminal_names
absorption_df = absorption_df.loc[adata_cr.obs_names, terminal_names]

# expected remaining time to *any* terminal state, weighted by which one this cell is fated for.
# absorption_time is `inf` wherever fate_probability is exactly 0 (never reaches that terminal) —
# guard the product explicitly, since 0 * inf = NaN in IEEE float rather than the mathematically
# correct "this state contributes nothing".
fate_arr = fate_df[terminal_names].values
absorption_arr = absorption_df.values
weighted = np.where(fate_arr == 0, 0.0, fate_arr * absorption_arr)
remaining_time = weighted.sum(axis=1)

# flip direction (root = furthest = 0, terminal = closest = 1) to match dpt_pseudotime's convention
pseudotime = 1.0 - (remaining_time - np.nanmin(remaining_time)) / (np.nanmax(remaining_time) - np.nanmin(remaining_time))

adata_cr.obs["pseudotime"] = pseudotime
adata_cr.obs["branch"] = adata_cr.obs[CLUSTER_KEY].astype("category")

adata_cr.uns["root_state"] = g.initial_states.cat.categories.tolist()[0]
adata_cr.uns["terminal_states"] = g.terminal_states.cat.categories.tolist()
adata_cr.uns["transient_states"] = [
    s
    for s in g.macrostates.cat.categories.tolist()
    if s not in adata_cr.uns["terminal_states"] and s != adata_cr.uns["root_state"]
]
adata_cr.uns["cellrank_pseudotime_method"] = "fate_probability_weighted_absorption_time"

print("root_state:", adata_cr.uns["root_state"])
print("terminal_states:", adata_cr.uns["terminal_states"])
print("transient_states:", adata_cr.uns["transient_states"])
print("NaN in pseudotime:", int(adata_cr.obs["pseudotime"].isna().sum()))
adata_cr.obs["pseudotime"].describe()

### Sanity check — pseudoczas rośnie korzeń→terminal?

Ten sam check co w TI notebooku: mediana `pseudotime` per `branch` (klaster) powinna rosnąć od `root_state` do `terminal_states`.

In [ ]:
sc.pl.embedding(
    adata_cr,
    basis="umap",
    color=["pseudotime", "branch"],
    wspace=0.4,
    show=True,
)

ax, order = plot_pseudotime_boxplot(
    adata_cr, ylabel="pseudotime (CellRank, opcja 4)"
)
ax.axhline(0, color="0.7", lw=0.8)
ax.axhline(1, color="0.7", lw=0.8)
ax.figure.tight_layout()
plt.show()

print("median pseudotime per branch (root → terminal expected low → high):")
print(order)

plot_pseudotime_kde(adata_cr)
plt.show()

### Sanity check #2 — porównanie z `scv.tl.velocity_pseudotime`

`velocity_pseudotime` (§9) jest lżejszym sanity-checkiem niż nasz `pseudotime` z opcji 4 — dyfuzja na skierowanym grafie velocity, nie fate-probability-weighted absorption time. Wysoka korelacja tu nie potwierdza, że *któryś* z nich jest "prawdziwy", tylko że są zgodne — to jest generyczny check (`plot_pseudotime_comparison` z `src/trajectory_inference/`, patrz `docs/chat-notes-veloBench.md` §18), przydatny na każdym datasecie/metodzie, nie tylko tu.

In [ ]:
from trajectory_inference import plot_pseudotime_comparison

scv.tl.velocity_graph(adata_cr)
scv.tl.velocity_pseudotime(adata_cr)

ax, rho = plot_pseudotime_comparison(
    adata_cr,
    "velocity_pseudotime",
    "pseudotime",
    x_label="scv.tl.velocity_pseudotime (sanity-check, §9)",
    y_label="pseudotime (CellRank opcja 4, §16)",
)
plt.show()
print(f"Spearman ρ = {rho:.3f}")

### Markery fenotypowe — dostępne, nieużyte tutaj

`plot_marker_genes` (`src/trajectory_inference/`) jest gotowa do użycia, ale Pancreas nie ma w tym repo swojego rejestru markerów (to co `states_and_markers` w `markers.py` daje dla HSC). Gdy pojawi się rejestr dla docelowego datasetu:

```python
from trajectory_inference import plot_marker_genes

plot_marker_genes(
    adata_cr,
    your_markers_dict,  # {state_name: [gene, ...]}
    priority_states=[adata_cr.uns["root_state"], *adata_cr.uns["terminal_states"], *adata_cr.uns["transient_states"]],
)
```

In [ ]:
CR_OUT_PATH = RESULTS_DIR / f"{DATASET}_cellrank_trajectory.h5ad"
adata_cr.write_h5ad(CR_OUT_PATH)
print(f"Wrote {CR_OUT_PATH}  (full: velocity layers + CellRank fate/absorption results)")

TRAJECTORY_PATH = RESULTS_DIR / "trajectory.h5ad"
ad_ti = write_trajectory_h5ad(adata_cr, TRAJECTORY_PATH, method=METHOD)
print(f"Wrote {TRAJECTORY_PATH}  (slim: TI contract only)")
print("obs:", list(ad_ti.obs.columns))
print("uns:", list(ad_ti.uns.keys()))
sorted(p.name for p in RESULTS_DIR.iterdir())

### Eksport w formacie TI (Q1–Q3, BoNesis, `trajectory.h5ad`)

Te same kroki co w `4. Trajectory Inference.ipynb` §12.6–12.8, teraz z `src/trajectory_inference/` (patrz `docs/chat-notes-veloBench.md` §15/§16) — więc oba pipeline'y (PAGA+DPT i CellRank) kończą się tym samym kontraktem plikowym, niezależnie od tego, że pseudoczas policzony jest inną metodą.

In [ ]:
RESULTS_DIR = ROOT / "results" / METHOD
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

observations = select_observations(adata_cr, pt_key="pseudotime", label_key="branch")
obs_csv = RESULTS_DIR / "pseudotime_inferred_observations.csv"
observations.to_frame().to_csv(obs_csv)

print(f"Wrote {obs_csv}  (n={len(observations)})")
observations.value_counts()

In [ ]:
constraints = derive_bonesis_constraints(
    adata_cr,
    observations,
    method=METHOD,
    pt_key="pseudotime",
    label_key="branch",
    terminal_states=adata_cr.uns["terminal_states"],
)
txt_path, json_path = write_bonesis_constraints(constraints, RESULTS_DIR)

print(f"Wrote {txt_path}")
print(f"Wrote {json_path}")
print("transient_chain:", constraints["transient_chain"])
print("terminals:", constraints["terminals"], "| hub:", constraints["hub"])
print("--- bonesis_lines ---")
for line in constraints["bonesis_lines"]:
    print(line)
constraints

In [ ]:
plot_state_graph(
    adata_cr,
    branch_key="branch",
    root_state=adata_cr.uns["root_state"],
    terminal_states=adata_cr.uns["terminal_states"],
)
plt.show()

### Co dalej

- `trajectory_edges` / `transient_order` — **zrobione**, `derive_bonesis_constraints` ustawia je jako side-effect w `adata_cr.uns`, tak jak w TI.
- `select_observations` / `derive_bonesis_constraints` / `write_trajectory_h5ad` — **wydzielone** do `src/trajectory_inference/` i używane przez oba notebooki (TI i tu); patrz `docs/chat-notes-veloBench.md` §17.
- `N_MACROSTATES=6` złapał 3 z 4 znanych terminal states na Pancreas (brakuje Delta/Epsilon) — do przestrojenia, jeśli te fenotypy są istotne.